# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、严谨的解释
- **额外要求**：用**流式（streaming）**一边生成一边更新显示，而不是等整段答完才一次性打印

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(..., stream=True)` |
| `messages`（system / user） | system 定角色；user 放 `question` |
| 流式输出 | 逐 chunk 拼接，并用 `update_display` 刷新 Markdown |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），用 `ollama.chat` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地需已安装并启动 Ollama，且已 `ollama pull llama3.2`
3. 在「提问」格改写 `question`，再分别跑 GPT 与 Llama 两格，对比回答风格


In [ ]:
# ========== 导入：环境、OpenAI、Jupyter 展示、dotenv ==========

# 导入标准库 os：读环境变量里的 OPENAI_API_KEY
import os
# 从 openai 导入 OpenAI 客户端：调用云端 Chat Completions
from openai import OpenAI
# Markdown/display/update_display：流式时先占位再不断刷新同一块输出
from IPython.display import Markdown, display, update_display
# load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv


In [ ]:
# ========== 常量：模型名字集中写在一处 ==========

# OpenAI 云端小模型：便宜、适合解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：须与 `ollama list` 一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境：加载 .env、检查密钥、创建客户端 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OpenAI 密钥（注意本练习用 OPENAI_API_KEY）
api_key=os.getenv("OPENAI_API_KEY")
# 简易体检：既不是 sk-proj- 前缀且长度很短 → 多半没配好
# 打印文案保持原样（含原作者拼写），避免改动可观察行为
if not api_key.startswith("sk-proj-") and len(api_key)<10:
    print("api key not foud")
else:
    print("api found and is ok")

# 创建默认 OpenAI 客户端（自动读环境变量中的密钥）
openai=OpenAI()
# 原作者留的空行打印，保持逻辑不变
print()


In [ ]:
# ========== 提问：改这里的 question 即可换题 ==========

# 三引号字符串：技术问题正文；发给模型的英文保持原样
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 流式调用 gpt-4o-mini，边生成边刷新 Markdown ==========

# messages：system 定角色 + user 放问题；content 字符串保持英文原样（含原拼写）
messages = [{"role":"system","content":"You are a expert Dta Scientist"}, {"role":"user","content":question}]

# stream=True：返回可迭代的增量 chunk，而不是一次性完整回复
stream = openai.chat.completions.create(
                model = MODEL_GPT,
                messages = messages,
                stream = True
)
# 累积已收到的文本
response = ""
# 先占位一块可更新的 Markdown 显示区
display_handle = display(Markdown(""), display_id=True)
# 遍历流式 chunk
for chunk in stream:
    # delta.content 可能为 None（例如结束片），用 '' 兜底
    response += chunk.choices[0].delta.content or ''
    # 去掉围栏标记，避免 update_display 时 Markdown 花屏（原逻辑保留）
    response = response.replace("```","").replace("markdown", "")
    # 用同一 display_id 刷新，实现「打字机」效果
    update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 用本地 Llama 3.2 流式回答（同一套 messages）==========

# 导入 ollama Python 包：调用本机 Ollama 服务
import ollama

# stream=True：按消息增量推送；model 用常量 MODEL_LLAMA
stream = ollama.chat(model=MODEL_LLAMA, messages=messages, stream=True)
# 累积回复文本
response = ""
# 新的可更新 Markdown 占位
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    # Ollama 流式块是 dict：从 message.content 取增量
    response += chunk["message"]["content"] or ''
    # 同样去掉代码围栏噪声
    response = response.replace("```","").replace("markdown", "")
    # 刷新同一块显示
    update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# （空单元格）可在此继续试验：换 question、换 model，或对比两家回答差异
